In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# BlindDetection-V1 engineering validation

Prepared but not executed. Select a GPU runtime and run once. The canary has no method conclusion; formal metrics are engineering observations only and science_denominator remains zero.

In [ ]:
from google.colab import userdata
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import sys
import torch
import uuid
import zipfile

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
PRODUCER_EXACT = '0a438ad2b322cdfc86ee91221027308702457fb1'
RUN_ID = f"blind-detection-v1-engineering-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid.uuid4().hex}"
CHECKOUT = Path('/content') / (RUN_ID + '-checkout')
LOCAL_ROOT = Path('/content') / (RUN_ID + '-local')
RUNTIME_ROOT = LOCAL_ROOT / 'runtime'
CURRENT_RGB_DIR = LOCAL_ROOT / 'current-rgb'
LOCAL_RESULT = LOCAL_ROOT / 'engineering_validation_result.json'
LOCAL_POSITIVE_ROWS = LOCAL_ROOT / 'engineering_positive_rows.json'
LOCAL_NEGATIVE_ROWS = LOCAL_ROOT / 'engineering_negative_rows.json'
LOCAL_STDOUT = LOCAL_ROOT / 'runner.stdout.txt'
LOCAL_STDERR = LOCAL_ROOT / 'runner.stderr.txt'
PUBLIC_CONFIG = CHECKOUT / 'configs/blind_detection/blind_detection_v1_engineering_validation.json'
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/engineering-validation-runs')
TERMINAL_ZIP = DRIVE_OUTPUT_ROOT / f'{RUN_ID}.zip'

if 'BLIND_ENGINEERING_RUNNER_CALLS' not in globals():
    BLIND_ENGINEERING_RUNNER_CALLS = 0
if 'BLIND_ENGINEERING_ZIP_WRITES' not in globals():
    BLIND_ENGINEERING_ZIP_WRITES = 0
if BLIND_ENGINEERING_RUNNER_CALLS != 0 or BLIND_ENGINEERING_ZIP_WRITES != 0:
    raise RuntimeError('this engineering validation execution cell is single-use')
if CHECKOUT.exists() or LOCAL_ROOT.exists() or TERMINAL_ZIP.exists():
    raise FileExistsError('fresh checkout, local root, and terminal ZIP path required')
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with LOCAL_STDOUT.open('xb'):
    pass
with LOCAL_STDERR.open('xb'):
    pass

def run_logged(command, *, cwd=None, env=None, check=True):
    with LOCAL_STDOUT.open('ab') as stdout, LOCAL_STDERR.open('ab') as stderr:
        return subprocess.run(
            command, cwd=cwd, env=env, stdout=stdout, stderr=stderr, check=check,
        )

def git_value(*args):
    return subprocess.run(
        ['git', '-C', str(CHECKOUT), *args],
        check=True, capture_output=True, text=True,
    ).stdout.strip()

def write_notebook_failure(stage, error):
    if not LOCAL_RESULT.exists():
        payload = {
            'automatic_retries': 0,
            'canary': {
                'method_conclusion': None,
                'method_scoring_performed': False,
                'operational_error': f'{type(error).__name__}: {error}',
                'science_denominator': 0,
                'status': 'OPERATIONAL_BLOCKED',
            },
            'claim_ceiling': 'engineering_observations_only_not_paper_ready_fpr_generalization_or_reliability',
            'error': f'{type(error).__name__}: {error}',
            'negative_denominator': 256,
            'negative_rows': [],
            'positive_denominator': 64,
            'positive_rows': [],
            'producer_exact': PRODUCER_EXACT,
            'science_denominator': 0,
            'stage': stage,
            'status': 'OPERATIONAL_BLOCKED',
            'success_criteria': None,
            'wrong_key_experiment': 'excluded',
        }
        with LOCAL_RESULT.open('xb') as sink:
            sink.write(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('ascii'))
    for path in (LOCAL_POSITIVE_ROWS, LOCAL_NEGATIVE_ROWS):
        if not path.exists():
            with path.open('xb') as sink:
                sink.write(b'[]')

def write_terminal_zip():
    global BLIND_ENGINEERING_ZIP_WRITES
    if BLIND_ENGINEERING_ZIP_WRITES != 0:
        raise RuntimeError('terminal ZIP publication may be attempted only once')
    BLIND_ENGINEERING_ZIP_WRITES += 1
    members = [
        (LOCAL_RESULT, LOCAL_RESULT.name),
        (LOCAL_POSITIVE_ROWS, LOCAL_POSITIVE_ROWS.name),
        (LOCAL_NEGATIVE_ROWS, LOCAL_NEGATIVE_ROWS.name),
        (LOCAL_STDOUT, LOCAL_STDOUT.name),
        (LOCAL_STDERR, LOCAL_STDERR.name),
    ]
    if checkout_verified and PUBLIC_CONFIG.is_file():
        members.append((PUBLIC_CONFIG, PUBLIC_CONFIG.name))
    if CURRENT_RGB_DIR.is_dir():
        members.extend(
            (path, 'current-rgb/' + path.name)
            for path in sorted(CURRENT_RGB_DIR.glob('*.png'))
        )
    with zipfile.ZipFile(TERMINAL_ZIP, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
        for source, arcname in members:
            archive.write(source, arcname=arcname)

stage = 'environment_guard'
runner_env = None
completed = None
checkout_verified = False
notebook_error = None
root_key = ''
hf_token = ''
try:
    if not torch.cuda.is_available():
        raise RuntimeError('GPU required; engineering validation was not executed')
    stage = 'detached_checkout'
    run_logged(['git', 'clone', REPO_URL, str(CHECKOUT)])
    run_logged(['git', '-C', str(CHECKOUT), 'checkout', '--detach', PRODUCER_EXACT])
    if git_value('rev-parse', 'HEAD') != PRODUCER_EXACT:
        raise RuntimeError('detached producer exact differs')
    if git_value('branch', '--show-current') != '' or git_value('status', '--porcelain=v1') != '':
        raise RuntimeError('producer checkout must be detached and clean')
    stage = 'dependency_install'
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', str(CHECKOUT)])
    if (
        git_value('rev-parse', 'HEAD') != PRODUCER_EXACT
        or git_value('branch', '--show-current') != ''
        or git_value('status', '--porcelain=v1') != ''
    ):
        raise RuntimeError('producer exact, detached state, or clean state changed during installation')
    checkout_verified = True
    stage = 'import_validation'
    run_logged([
        sys.executable, '-c',
        'from experiments import run_blind_detection_v1 as r; r.load_engineering_validation_config(r.REPO_ROOT); r.load_repository_threshold_asset(r.REPO_ROOT)',
    ], cwd=CHECKOUT)
    stage = 'secret_validation'
    root_key = userdata.get('CEG_WM_ROOT_KEY')
    hf_token = userdata.get('HF_TOKEN')
    if not isinstance(root_key, str) or not root_key.strip():
        raise RuntimeError('CEG_WM_ROOT_KEY Colab Secret is required')
    if not isinstance(hf_token, str) or not hf_token.strip():
        raise RuntimeError('HF_TOKEN Colab Secret is required')
    secret_markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
    runner_env = {
        name: value for name, value in os.environ.items()
        if not any(marker in name.upper() for marker in secret_markers)
    }
    runner_env['CEG_WM_ROOT_KEY'] = root_key
    runner_env['HF_TOKEN'] = hf_token
    root_key = ''
    hf_token = ''
    stage = 'formal_runner'
    command = [
        sys.executable, '-m', 'experiments.run_blind_detection_v1', 'engineering-validate',
        '--producer-exact', PRODUCER_EXACT,
        '--runtime-root', str(RUNTIME_ROOT),
        '--current-rgb-output-dir', str(CURRENT_RGB_DIR),
        '--result-output', str(LOCAL_RESULT),
        '--positive-rows-output', str(LOCAL_POSITIVE_ROWS),
        '--negative-rows-output', str(LOCAL_NEGATIVE_ROWS),
    ]
    BLIND_ENGINEERING_RUNNER_CALLS += 1
    completed = run_logged(command, cwd=CHECKOUT, env=runner_env, check=False)
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    runner_env = None
    if BLIND_ENGINEERING_RUNNER_CALLS != 1 or not LOCAL_RESULT.is_file():
        raise RuntimeError('formal engineering validation result is absent')
    stage = 'terminal_zip_publication'
except BaseException as error:
    notebook_error = error
finally:
    root_key = ''
    hf_token = ''
    if runner_env is not None:
        runner_env.pop('CEG_WM_ROOT_KEY', None)
        runner_env.pop('HF_TOKEN', None)
    runner_env = None
    if notebook_error is not None:
        write_notebook_failure(stage, notebook_error)
    if not LOCAL_RESULT.is_file():
        write_notebook_failure(stage, RuntimeError('engineering validation result is absent'))
    write_terminal_zip()

public_result = json.loads(LOCAL_RESULT.read_text(encoding='ascii'))
print('CEGWM_BLIND_ENGINEERING_PUBLISHED ' + json.dumps({
    'negative_denominator': public_result.get('negative_denominator'),
    'positive_denominator': public_result.get('positive_denominator'),
    'runner_returncode': None if completed is None else completed.returncode,
    'science_denominator': public_result.get('science_denominator'),
    'status': public_result.get('status'),
    'terminal_zip': str(TERMINAL_ZIP),
}, sort_keys=True))


In [ ]:
with zipfile.ZipFile(TERMINAL_ZIP, mode='r') as archive:
    public_result = json.loads(
        archive.read('engineering_validation_result.json').decode('ascii')
    )
    print('CEGWM_BLIND_ENGINEERING_READBACK ' + json.dumps({
        'negative_denominator': public_result.get('negative_denominator'),
        'positive_denominator': public_result.get('positive_denominator'),
        'science_denominator': public_result.get('science_denominator'),
        'status': public_result.get('status'),
    }, sort_keys=True))
